In [7]:
!pip install scikit-learn
import sys
from pathlib import Path
import pandas as pd
print(pd.__version__)  

sys.path.insert(0, str(Path("..").resolve()))
sys.path.insert(0, "../Prop_repo/Property-based-Neural-Network-Verification/src")
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, n_features: int, num_classes: int):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = x.squeeze(1)
        return self.head(x)

import utils.models
utils.models.MLP = MLP

print(utils.models)
print(hasattr(utils.models, "MLP"))
print(utils.models.MLP)

import joblib
bundle = joblib.load("model.joblib")
print(bundle.keys())
print(type(bundle["model"]))
print(bundle.keys())

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 9.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 10.8 MB/s  0:00:01 eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
3.0.0
<module 'utils.models' from '/Users/karlivarkvaran/Desktop/Attacks_on_models/utils/models.py'>
True
<class '__main__.MLP'>
dict_keys(['model', 'features', 'labels', 'model_type', 'scaler', 'scale_cols', 'clip_lower', 'clip_upper', 'config'])
<class 'thesis.models.torch_models.MLP'>
dict_keys(['model', 'features', 'labels', 'model_type', 'scaler', 'scale_cols', 'clip_lower', 'clip_upper', 'config'])


In [8]:

import torch
import torch.nn as nn

base_model = bundle["model"]
base_model.eval()

class FlatMLPWrapper(nn.Module):
    def __init__(self, mlp):
        super().__init__()
        self.mlp = mlp

    def forward(self, x):
        return self.mlp(x)

wrapped_model = FlatMLPWrapper(base_model).eval()

n_features = len(bundle["features"])
dummy_input = torch.randn(1, n_features)

torch.onnx.export(
    wrapped_model,
    dummy_input,
    "model_pip.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=11,
    dynamo=False
)

/var/folders/5b/8cyxhz850xlg6xrjw4tswf4h0000gn/T/ipykernel_48430/1464612471.py:20: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [9]:

from maraboupy import Marabou

net = Marabou.read_onnx("formal_moda_best.onnx")


ModuleNotFoundError: No module named 'maraboupy'

In [ ]:
print(bundle)

classes = ["BENIGN", "HTTP_FLOOD", "PORTSCAN"]


In [ ]:
classes = ["BENIGN", "HTTP_FLOOD", "PORTSCAN"]



features = list(bundle["features"])
for i, f in enumerate(features):
    print(i, f)

In [ ]:
ATTACK_SPECS = {
    "validity": {
        "valid_packet_size_min_pkts": 1.0,
        "valid_packet_size_min_total_bytes": 40.0,
    },
    "dos_http_flood": {
        "mal_time_elapsed_min": 0.0,
        "mal_time_elapsed_max": 120.0,
        "valid_pkt_size_total_min": 250,
        "mal_byte_rate_min": 200.0,
        "mal_pkt_rate_min": 5.0,
    },
    "portscan": {
        "min_uniq_dst_ports": 20.0,
        "max_pkts_per_port": 5.0,
        "max_scan_duration": 1.0,
        "min_fail_ratio": 0.75,
    },
    "ddos_udp_flood": {
        "max_udp_duration": 10.0,
        "min_udp_conn_count": 10.0,
        "min_udp_packets": 50.0,
        "min_udp_rate": 10.0,
        "min_unique_src_ips": 1.0,
    },
    "ddos_syn_flood": {
        "max_syn_duration": 10.0,
        "min_syn_conn_count": 5.0,
        "min_syn_count": 5.0,
        "min_syn_rate": 1.0,
        "min_half_open_count": 1.0,
        "min_source_ip_count": 1.0,
    },
} 

In [ ]:
RAW_BOUNDS = {
    "duration": (0.0, 120.0),
    "orig_bytes": (0.0, 200_000.0),
    "resp_bytes": (0.0, 50_000.0),
    "missed_bytes": (0.0, 1000.0),
    "orig_pkts": (0.0, 10_000.0),
    "orig_ip_bytes": (0.0, 500_000.0),
    "resp_pkts": (0.0, 5000.0),
    "resp_ip_bytes": (0.0, 200_000.0),
    "orig_pkt_rate": (0.0, 50_000.0),
    "orig_byte_rate": (0.0, 5_000_000.0),
    "time_elapsed": (0.0, 120.0),
    "valid_tcp_handshake": (0.0, 1.0),
    "valid_http_conn": (0.0, 1.0),
    "uniq_dst_ports": (1.0, 5000.0),
    "pkts_per_port": (0.0, 10_000.0),
    "scan_duration": (0.0, 120.0),
    "fail_ratio": (0.0, 1.0),
    "id.resp_p": (0.0, 65535.0),
}

In [ ]:
def verify_portscan_property(
    model_path="formal_moda_best.onnx",
    rival_classes=("BENIGN", "DOS_HTTP_FLOOD"),
    eps=1e-5,
    print_counterexample=True,
):
    from maraboupy import Marabou

    classes = ["BENIGN", "DOS_HTTP_FLOOD", "PORTSCAN"]
    features = list(bundle["features"])

    portscan_idx = classes.index("PORTSCAN")
    idx = {name: features.index(name) for name in features}

    for rival_name in rival_classes:
        rival_idx = classes.index(rival_name)
        print(f"Checking rival class: {rival_name}")

        net = Marabou.read_onnx(model_path)
        inputs = net.inputVars[0][0]
        output_vars = net.outputVars[0][0]

        # --- Broad base bounds ---
        for i, feat in enumerate(features):
            raw_lb, raw_ub = RAW_BOUNDS[feat]
            net.setLowerBound(inputs[i], float(raw_lb))
            net.setUpperBound(inputs[i], float(raw_ub))

        # --- Timing ---
        net.setLowerBound(inputs[idx["duration"]], 0.5)
        net.setUpperBound(inputs[idx["duration"]], 4.0)

        net.setLowerBound(inputs[idx["time_elapsed"]], 0.5)
        net.setUpperBound(inputs[idx["time_elapsed"]], 4.0)

        net.setLowerBound(inputs[idx["scan_duration"]], 0.1)
        net.setUpperBound(inputs[idx["scan_duration"]], 0.5)

        # --- Timing consistency ---
        # time_elapsed >= duration
        net.addInequality(
            [int(inputs[idx["duration"]]), int(inputs[idx["time_elapsed"]])],
            [1.0, -1.0],
            0.0,
        )
        # time_elapsed <= 2 * duration
        net.addInequality(
            [int(inputs[idx["time_elapsed"]]), int(inputs[idx["duration"]])],
            [1.0, -2.0],
            0.0,
        )
        # scan_duration <= duration
        net.addInequality(
            [int(inputs[idx["scan_duration"]]), int(inputs[idx["duration"]])],
            [1.0, -1.0],
            0.0,
        )
        # scan_duration >= 0.5 * duration
        net.addInequality(
            [int(inputs[idx["scan_duration"]]), int(inputs[idx["duration"]])],
            [-1.0, 0.5],
            0.0,
        )

        # --- Scan behavior flags ---
        net.setLowerBound(inputs[idx["valid_tcp_handshake"]], 0.0)
        net.setUpperBound(inputs[idx["valid_tcp_handshake"]], 0.0)

        net.setLowerBound(inputs[idx["valid_http_conn"]], 0.0)
        net.setUpperBound(inputs[idx["valid_http_conn"]], 0.0)

        net.setLowerBound(inputs[idx["fail_ratio"]], 0.95)
        net.setUpperBound(inputs[idx["fail_ratio"]], 1.0)

        # --- Port diversity ---
        net.setLowerBound(inputs[idx["uniq_dst_ports"]], 50.0)
        net.setUpperBound(inputs[idx["uniq_dst_ports"]], 100.0)

        net.setLowerBound(inputs[idx["pkts_per_port"]], 1.0)
        net.setUpperBound(inputs[idx["pkts_per_port"]], 3.0)

        # orig_pkts >= uniq_dst_ports (at min 1 pkt/port)
        net.addInequality(
            [int(inputs[idx["orig_pkts"]]), int(inputs[idx["uniq_dst_ports"]])],
            [-1.0, 1.0],
            0.0,
        )
        # orig_pkts <= 3 * uniq_dst_ports (at max 3 pkts/port)
        net.addInequality(
            [int(inputs[idx["orig_pkts"]]), int(inputs[idx["uniq_dst_ports"]])],
            [1.0, -3.0],
            0.0,
        )

        # --- Packet volume ---
        net.setLowerBound(inputs[idx["orig_pkts"]], 50.0)
        net.setUpperBound(inputs[idx["orig_pkts"]], 300.0)

        # orig_pkt_rate: just a box, no envelopes — derived feature, can't
        # be reliably constrained relative to orig_pkts/duration without
        # introducing exploitable gaps
        net.setLowerBound(inputs[idx["orig_pkt_rate"]], 100.0)
        net.setUpperBound(inputs[idx["orig_pkt_rate"]], 100.0)

        # --- Byte volume ---
        net.setLowerBound(inputs[idx["orig_bytes"]], 2000.0)
        net.setUpperBound(inputs[idx["orig_bytes"]], 30000.0)

        # orig_bytes in [40*pkts, 100*pkts] — SYN probes are 40-100 bytes
        net.addInequality(
            [int(inputs[idx["orig_bytes"]]), int(inputs[idx["orig_pkts"]])],
            [-1.0, 40.0],
            0.0,
        )
        net.addInequality(
            [int(inputs[idx["orig_bytes"]]), int(inputs[idx["orig_pkts"]])],
            [1.0, -100.0],
            0.0,
        )

        # orig_ip_bytes >= orig_bytes + 1000 (min IP header overhead)
        net.setLowerBound(inputs[idx["orig_ip_bytes"]], 3000.0)
        net.setUpperBound(inputs[idx["orig_ip_bytes"]], 36000.0)
        net.addInequality(
            [int(inputs[idx["orig_ip_bytes"]]), int(inputs[idx["orig_bytes"]])],
            [-1.0, 1.0],
            -1000.0,
        )

        # orig_byte_rate: just a box, no envelopes — same reason as orig_pkt_rate
        net.setLowerBound(inputs[idx["orig_byte_rate"]], 10000.0)
        net.setUpperBound(inputs[idx["orig_byte_rate"]], 10000.0)

        # --- Response traffic ---
        net.setLowerBound(inputs[idx["resp_pkts"]], 0.0)
        net.setUpperBound(inputs[idx["resp_pkts"]], 20.0)

        net.setLowerBound(inputs[idx["resp_bytes"]], 0.0)
        net.setUpperBound(inputs[idx["resp_bytes"]], 1000.0)

        net.setLowerBound(inputs[idx["resp_ip_bytes"]], 0.0)
        net.setUpperBound(inputs[idx["resp_ip_bytes"]], 2000.0)

        # resp_ip_bytes in [56*pkts, 100*pkts] (ICMP unreachable/RST size range)
        net.addInequality(
            [int(inputs[idx["resp_ip_bytes"]]), int(inputs[idx["resp_pkts"]])],
            [-1.0, 56.0],
            0.0,
        )
        net.addInequality(
            [int(inputs[idx["resp_ip_bytes"]]), int(inputs[idx["resp_pkts"]])],
            [1.0, -100.0],
            0.0,
        )
        # resp_bytes <= resp_ip_bytes
        net.addInequality(
            [int(inputs[idx["resp_bytes"]]), int(inputs[idx["resp_ip_bytes"]])],
            [1.0, -1.0],
            0.0,
        )
        # If fail_ratio is high, resp_bytes should be near zero
        # RST/ICMP unreachable packets have no payload
        # resp_bytes <= (1 - fail_ratio) * resp_pkts * 100
        # Can't encode directly (nonlinear), but since fail_ratio >= 0.95:
        # resp_bytes <= 0.05 * resp_pkts * 100 = 5 * resp_pkts
        net.addInequality(
            [int(inputs[idx["resp_bytes"]]), int(inputs[idx["resp_pkts"]])],
            [1.0, -5.0],   # resp_bytes - 5*resp_pkts <= 0
            0.0,
)

        # --- Noise floor ---
        net.setLowerBound(inputs[idx["missed_bytes"]], 0.0)
        net.setUpperBound(inputs[idx["missed_bytes"]], 10.0)

        # --- Property: can rival beat PORTSCAN? ---
        net.addInequality(
            [int(output_vars[portscan_idx]), int(output_vars[rival_idx])],
            [1.0, -1.0],
            -eps,
        )

        exit_code, vals, stats = net.solve()

        if exit_code == "sat" or (isinstance(vals, dict) and len(vals) > 0):
            print(f"Counterexample found: {rival_name} beats PORTSCAN")

            if print_counterexample:
                print("\nInput feature values:")
                for i, name in enumerate(features):
                    print(f"{i:2d} {name:30s} raw={vals.get(inputs[i], None)!r}")
                d   = vals.get(inputs[idx["duration"]], None)
                op  = vals.get(inputs[idx["orig_pkts"]], None)
                ob  = vals.get(inputs[idx["orig_bytes"]], None)
                r   = vals.get(inputs[idx["orig_pkt_rate"]], None)
                br  = vals.get(inputs[idx["orig_byte_rate"]], None)
                oip = vals.get(inputs[idx["orig_ip_bytes"]], None)
                rp  = vals.get(inputs[idx["resp_pkts"]], None)
                rip = vals.get(inputs[idx["resp_ip_bytes"]], None)
                if None not in (d, op, r) and d > 0:
                    print(f"  [sanity] orig_pkt_rate  expected={op/d:.2f}  got={r:.2f}  gap={abs(r - op/d):.2f}")
                if None not in (d, ob, br) and d > 0:
                    print(f"  [sanity] orig_byte_rate expected={ob/d:.2f}  got={br:.2f}  gap={abs(br - ob/d):.2f}")
                if None not in (ob, oip):
                    print(f"  [sanity] orig_ip_bytes  expected>={ob:.2f}+1000  got={oip:.2f}  ok={oip >= ob + 1000}")
                if None not in (rp, rip):
                    print(f"  [sanity] resp_ip_bytes  expected>={rp*56:.2f}  got={rip:.2f}  ok={rip >= rp * 56}")
                print("\nOutput values:")
                for i, cls_name in enumerate(classes):
                    print(f"{i:2d} {cls_name:20s} logit={vals.get(output_vars[i], None)!r}")

            return False, rival_name, vals, stats

        elif exit_code == "unsat":
            print(f"No counterexample for rival {rival_name}")

        else:
            print(f"Solver returned: {exit_code}")

    print("No counterexample found. Property holds against all selected rival classes.")
    return True, None, None, None

In [ ]:
verify_portscan_property(
    model_path="formal_moda_best.onnx",
    rival_classes=("BENIGN", "DOS_HTTP_FLOOD"),
    eps = 0.1,
)

In [ ]:
def verify_http_flood_property(
    model_path="formal_moda_best.onnx",
    rival_classes=("BENIGN", "PORTSCAN"),
    eps=0.5,
    print_counterexample=True,
):
    from maraboupy import Marabou

    classes = ["BENIGN", "DOS_HTTP_FLOOD", "PORTSCAN"]
    features = list(bundle["features"])

    http_idx = classes.index("DOS_HTTP_FLOOD")
    idx = {name: features.index(name) for name in features}

    for rival_name in rival_classes:
        rival_idx = classes.index(rival_name)
        print(f"Checking rival class: {rival_name}")

        net = Marabou.read_onnx(model_path)
        inputs = net.inputVars[0][0]
        output_vars = net.outputVars[0][0]

        # --- Broad base bounds ---
        for i, feat in enumerate(features):
            raw_lb, raw_ub = RAW_BOUNDS[feat]
            net.setLowerBound(inputs[i], float(raw_lb))
            net.setUpperBound(inputs[i], float(raw_ub))

        # --- Timing ---
        net.setLowerBound(inputs[idx["duration"]], 1.0)
        net.setUpperBound(inputs[idx["duration"]], 10.0)

        net.setLowerBound(inputs[idx["time_elapsed"]], 1.0)
        net.setUpperBound(inputs[idx["time_elapsed"]], 10.0)

        net.setLowerBound(inputs[idx["scan_duration"]], 0.0)
        net.setUpperBound(inputs[idx["scan_duration"]], 0.05)

        # --- Timing consistency ---
        # time_elapsed >= duration
        net.addInequality(
            [int(inputs[idx["duration"]]), int(inputs[idx["time_elapsed"]])],
            [1.0, -1.0],
            0.0,
        )
        # time_elapsed <= 2 * duration
        net.addInequality(
            [int(inputs[idx["time_elapsed"]]), int(inputs[idx["duration"]])],
            [1.0, -2.0],
            0.0,
        )
        # scan_duration <= duration
        net.addInequality(
            [int(inputs[idx["scan_duration"]]), int(inputs[idx["duration"]])],
            [1.0, -1.0],
            0.0,
        )

        # --- HTTP flood behavior flags ---
        net.setLowerBound(inputs[idx["valid_tcp_handshake"]], 1.0)
        net.setUpperBound(inputs[idx["valid_tcp_handshake"]], 1.0)

        net.setLowerBound(inputs[idx["valid_http_conn"]], 1.0)
        net.setUpperBound(inputs[idx["valid_http_conn"]], 1.0)

        net.setLowerBound(inputs[idx["fail_ratio"]], 0.0)
        net.setUpperBound(inputs[idx["fail_ratio"]], 0.05)

        # --- Port behavior ---
        net.setLowerBound(inputs[idx["uniq_dst_ports"]], 1.0)
        net.setUpperBound(inputs[idx["uniq_dst_ports"]], 2.0)

        net.setLowerBound(inputs[idx["pkts_per_port"]], 10.0)
        net.setUpperBound(inputs[idx["pkts_per_port"]], 200.0)

        # --- Packet volume ---
        net.setLowerBound(inputs[idx["orig_pkts"]], 200.0)
        net.setUpperBound(inputs[idx["orig_pkts"]], 10000.0)

        # orig_pkts >= uniq_dst_ports * min_pkts_per_port (10)
        net.addInequality(
            [int(inputs[idx["orig_pkts"]]), int(inputs[idx["uniq_dst_ports"]])],
            [-1.0, 10.0],
            0.0,
        )
        # orig_pkts <= uniq_dst_ports * max_pkts_per_port (200)
        net.addInequality(
            [int(inputs[idx["orig_pkts"]]), int(inputs[idx["uniq_dst_ports"]])],
            [1.0, -200.0],
            0.0,
        )

        # pkts_per_port = orig_pkts / uniq_dst_ports
        # Lower: pkts_per_port >= orig_pkts / max_ports (2)  =>  2*pkts_per_port >= orig_pkts
        # => pkts_per_port - 0.5*orig_pkts >= 0
        net.addInequality(
            [int(inputs[idx["pkts_per_port"]]), int(inputs[idx["orig_pkts"]])],
            [-1.0, 0.5],   # -pkts_per_port + 0.5*orig_pkts <= 0
            0.0,
        )
        # Upper: pkts_per_port <= orig_pkts / min_ports (1)  =>  pkts_per_port <= orig_pkts
        # => pkts_per_port - orig_pkts <= 0
        net.addInequality(
            [int(inputs[idx["pkts_per_port"]]), int(inputs[idx["orig_pkts"]])],
            [1.0, -1.0],   # pkts_per_port - orig_pkts <= 0
            0.0,
        )

        # orig_pkt_rate: box only — derived feature
        net.setLowerBound(inputs[idx["orig_pkt_rate"]], 100.0)
        net.setUpperBound(inputs[idx["orig_pkt_rate"]], 100.0)

        # --- Byte volume ---
        net.setLowerBound(inputs[idx["orig_bytes"]], 5000.0)
        net.setUpperBound(inputs[idx["orig_bytes"]], 200000.0)

        # HTTP requests are 100-1500 bytes each
        net.addInequality(
            [int(inputs[idx["orig_bytes"]]), int(inputs[idx["orig_pkts"]])],
            [-1.0, 100.0],
            0.0,
        )
        net.addInequality(
            [int(inputs[idx["orig_bytes"]]), int(inputs[idx["orig_pkts"]])],
            [1.0, -1500.0],
            0.0,
        )

        # orig_ip_bytes >= orig_bytes + 1000
        net.setLowerBound(inputs[idx["orig_ip_bytes"]], 5000.0)
        net.setUpperBound(inputs[idx["orig_ip_bytes"]], 500000.0)
        net.addInequality(
            [int(inputs[idx["orig_ip_bytes"]]), int(inputs[idx["orig_bytes"]])],
            [-1.0, 1.0],
            -1000.0,
        )

        # orig_byte_rate: box only — derived feature
        net.setLowerBound(inputs[idx["orig_byte_rate"]], 10000.0)
        net.setUpperBound(inputs[idx["orig_byte_rate"]], 10000.0)

        # --- Response traffic ---
        net.setLowerBound(inputs[idx["resp_pkts"]], 0.0)
        net.setUpperBound(inputs[idx["resp_pkts"]], 10000.0)

        # resp_pkts >= orig_pkts * 0.5
        net.addInequality(
            [int(inputs[idx["resp_pkts"]]), int(inputs[idx["orig_pkts"]])],
            [-1.0, 0.5],
            0.0,
        )

        net.setLowerBound(inputs[idx["resp_bytes"]], 0.0)
        net.setUpperBound(inputs[idx["resp_bytes"]], 50000.0)

        net.setLowerBound(inputs[idx["resp_ip_bytes"]], 0.0)
        net.setUpperBound(inputs[idx["resp_ip_bytes"]], 50000.0)

        # resp_bytes in [200*resp_pkts, 1500*resp_pkts] — HTTP responses are 200+ bytes
        net.addInequality(
            [int(inputs[idx["resp_bytes"]]), int(inputs[idx["resp_pkts"]])],
            [-1.0, 200.0],   # resp_bytes >= 200 * resp_pkts
            0.0,
        )
        net.addInequality(
            [int(inputs[idx["resp_bytes"]]), int(inputs[idx["resp_pkts"]])],
            [1.0, -1500.0],  # resp_bytes <= 1500 * resp_pkts
            0.0,
        )

        # resp_ip_bytes >= resp_bytes + 500 (IP + TCP header overhead)
        net.addInequality(
            [int(inputs[idx["resp_ip_bytes"]]), int(inputs[idx["resp_bytes"]])],
            [-1.0, 1.0],
            -500.0,
        )
        # resp_ip_bytes <= 1500 * resp_pkts (max ethernet frame)
        net.addInequality(
            [int(inputs[idx["resp_ip_bytes"]]), int(inputs[idx["resp_pkts"]])],
            [1.0, -1500.0],
            0.0,
        )

        # --- Noise floor ---
        net.setLowerBound(inputs[idx["missed_bytes"]], 0.0)
        net.setUpperBound(inputs[idx["missed_bytes"]], 10.0)

        # --- Property: can rival beat DOS_HTTP_FLOOD? ---
        net.addInequality(
            [int(output_vars[http_idx]), int(output_vars[rival_idx])],
            [1.0, -1.0],
            -eps,
        )

        exit_code, vals, stats = net.solve()

        if exit_code == "sat" or (isinstance(vals, dict) and len(vals) > 0):
            print(f"Counterexample found: {rival_name} beats DOS_HTTP_FLOOD")

            if print_counterexample:
                print("\nInput feature values:")
                for i, name in enumerate(features):
                    print(f"{i:2d} {name:30s} raw={vals.get(inputs[i], None)!r}")
                d   = vals.get(inputs[idx["duration"]], None)
                op  = vals.get(inputs[idx["orig_pkts"]], None)
                ob  = vals.get(inputs[idx["orig_bytes"]], None)
                r   = vals.get(inputs[idx["orig_pkt_rate"]], None)
                br  = vals.get(inputs[idx["orig_byte_rate"]], None)
                oip = vals.get(inputs[idx["orig_ip_bytes"]], None)
                rp  = vals.get(inputs[idx["resp_pkts"]], None)
                rib = vals.get(inputs[idx["resp_ip_bytes"]], None)
                rb  = vals.get(inputs[idx["resp_bytes"]], None)
                ppp = vals.get(inputs[idx["pkts_per_port"]], None)
                udp = vals.get(inputs[idx["uniq_dst_ports"]], None)
                if None not in (d, op, r) and d > 0:
                    print(f"  [sanity] orig_pkt_rate  expected={op/d:.2f}  got={r:.2f}  gap={abs(r - op/d):.2f}")
                if None not in (d, ob, br) and d > 0:
                    print(f"  [sanity] orig_byte_rate expected={ob/d:.2f}  got={br:.2f}  gap={abs(br - ob/d):.2f}")
                if None not in (ob, oip):
                    print(f"  [sanity] orig_ip_bytes  expected>={ob:.2f}+1000  got={oip:.2f}  ok={oip >= ob + 1000}")
                if None not in (rb, rib):
                    print(f"  [sanity] resp_ip_bytes  expected>={rb:.2f}+500  got={rib:.2f}  ok={rib >= rb + 500}")
                if None not in (rp, op):
                    print(f"  [sanity] resp_pkts      expected>={op*0.5:.2f}  got={rp:.2f}  ok={rp >= op * 0.5}")
                if None not in (ppp, op, udp) and udp > 0:
                    print(f"  [sanity] pkts_per_port  expected={op/udp:.2f}  got={ppp:.2f}  gap={abs(ppp - op/udp):.2f}")
                print("\nOutput values:")
                for i, cls_name in enumerate(classes):
                    print(f"{i:2d} {cls_name:20s} logit={vals.get(output_vars[i], None)!r}")

            return False, rival_name, vals, stats

        elif exit_code == "unsat":
            print(f"No counterexample for rival {rival_name}")

        else:
            print(f"Solver returned: {exit_code}")

    print("No counterexample found. Property holds.")
    return True, None, None, None

In [ ]:
verify_http_flood_property(
    model_path="formal_moda_best.onnx",
    rival_classes=("BENIGN","PORTSCAN"),
    eps=1e-5,
)